In [3]:
from pathlib import Path
from html import escape
import json
import sys
from time import perf_counter
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/indexes/sus/bm25.json").is_file()
             and (p / "src/document_pipeline/retrieval.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Buka notebook dari proyek RAG. Indeks data/indexes/sus/bm25.json harus tersedia.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.document_pipeline.retrieval import search

INDEX_PATH = ROOT / "data/indexes/sus/bm25.json"
index_info = json.loads(INDEX_PATH.read_text(encoding="utf-8"))
print(f"BM25 lokal | {index_info['document_count']} chunk | k1={index_info['parameters']['k1']} | b={index_info['parameters']['b']}")
print("Indeks:", INDEX_PATH)


BM25 lokal | 16 chunk | k1=1.5 | b=0.75
Indeks: c:\Users\Maulana\Documents\Project\Intern\RAG\data\indexes\sus\bm25.json


In [4]:
query_input = widgets.Text(
    description="Query:", placeholder="Contoh: 120 respondents questionnaire",
    continuous_update=False, layout=widgets.Layout(width="95%"))
top_k = widgets.BoundedIntText(value=3, min=1, max=index_info["document_count"], description="Top-k:")
search_button = widgets.Button(description="Cari BM25", icon="search", button_style="primary")
status_view = widgets.HTML(value="Masukkan query, lalu tekan Enter atau klik Cari BM25.")
result_output = widgets.Output()
json_view = widgets.Textarea(disabled=True, layout=widgets.Layout(width="100%", height="450px"))
result_tabs = widgets.Tab(children=[result_output, json_view])
result_tabs.set_title(0, "Hasil pencarian")
result_tabs.set_title(1, "JSON hasil")
last_result = None


def html_table(headers, rows):
    def cells(values, tag):
        return "".join(f"<{tag} style='border:1px solid #888;padding:7px;text-align:left'>{escape(str(v))}</{tag}>" for v in values)
    return ("<div style='overflow-x:auto'><table style='border-collapse:collapse'>"
            + "<thead><tr>" + cells(headers, "th") + "</tr></thead><tbody>"
            + "".join("<tr>" + cells(row, "td") + "</tr>" for row in rows)
            + "</tbody></table></div>")


def run_query(event=None):
    global last_result
    query = query_input.value.strip()
    last_result = None
    json_view.value = ""
    with result_output:
        clear_output(wait=False)
    if not query:
        status_view.value = "<b>Masukkan query terlebih dahulu.</b>"
        return
    search_button.disabled = True
    status_view.value = "Mencari..."
    try:
        started = perf_counter()
        # Fungsi retrieval yang sama dengan CLI; hash korpus diverifikasi tiap pencarian.
        result = search(INDEX_PATH, query, top_k.value)
        elapsed_ms = (perf_counter() - started) * 1000
        last_result = result
        json_view.value = json.dumps(result, ensure_ascii=False, indent=2)
        hits = result["results"]
        status_view.value = (f"<b>{len(hits)} hasil</b> | {elapsed_ms:.1f} ms | Token query: "
                             + escape(", ".join(result["query_tokens"]))
                             + "<br>Skor BM25 bukan probabilitas. Status artefak ditampilkan sesuai JSON sumber.")
        with result_output:
            if not hits:
                display(HTML("<p>Tidak ada token query yang cocok. Coba istilah lain; dokumen ini berbahasa Inggris.</p>"))
                return
            display(HTML(html_table(["Rank", "Skor BM25", "Section", "Halaman PDF", "Chunk ID"], [
                [h["rank"], f"{h['score']:.6f}", " > ".join(h["section_path"]) or "Pembuka", h["pages"], h["chunk_id"]]
                for h in hits])))
            panels = []
            for hit in hits:
                details = [("Chunk ID",hit["chunk_id"]), ("Sumber",(hit.get("source") or {}).get("name", "")),
                           ("Path sumber",(hit.get("source") or {}).get("path", "")),
                           ("Halaman PDF",hit["pages"]), ("Matched terms",", ".join(hit["matched_terms"])),
                           ("Jenis konten",hit["content_types"]), ("Status artefak",hit.get("status")),
                           ("Aset",hit.get("asset_refs",[]))]
                info = widgets.HTML(value="<br>".join(f"<b>{escape(k)}:</b> {escape(str(v))}" for k,v in details))
                # Textarea menampilkan seluruh teks, tanpa memotong chunk panjang.
                full_text = widgets.Textarea(value=hit["text"], disabled=True,
                                             layout=widgets.Layout(width="100%", height="380px"))
                panels.append(widgets.VBox([info, full_text]))
            accordion = widgets.Accordion(children=panels, selected_index=0)
            for i,hit in enumerate(hits):
                section = " > ".join(hit["section_path"]) or "Pembuka"
                accordion.set_title(i, f"{hit['rank']}. {section} | skor {hit['score']:.4f}")
            display(accordion)
    except (OSError, ValueError, KeyError, TypeError) as error:
        status_view.value = "<b>Pencarian gagal:</b> " + escape(str(error))
        with result_output:
            print("Jika JSON chunk berubah, bangun ulang indeks BM25 melalui main.py index-bm25 dengan --overwrite.")
    finally:
        search_button.disabled = False


search_button.on_click(run_query)
query_input.observe(run_query, names="value")
display(widgets.VBox([query_input, widgets.HBox([top_k, search_button]), status_view, result_tabs]))
